# Backtesting Strategies
Test SMA Crossover, RSI Mean Reversion, and Bollinger Band Breakout on BTC 5m data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import vectorbt as vbt
from pathlib import Path
import sys

%matplotlib inline

# Ensure scripts are importable from notebook directory
_scripts = Path("/mnt/c/Users/revno/PROJECTS/CLAUDE/workspace/scripts")
sys.path.insert(0, str(_scripts))
from load_data import btc, resample

OUT = (Path.cwd().parent / "outputs").resolve()
OUT.mkdir(exist_ok=True)

## Load & Prepare Data
Using 1h aggregation for smoother signals

In [ ]:
raw = btc()
df = resample(raw, "1h")

close = df["close"]
high = df["high"]
low = df["low"]

print(f"Bars: {len(close):,}")
print(f"Range: {close.index[0]} to {close.index[-1]}")
print(f"Close: ${close.iloc[0]:,.2f} → ${close.iloc[-1]:,.2f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
fig.suptitle("BTC/USDT — 1h Aggregated (5m source) | UTC+8", fontsize=14)

axes[0].plot(close.index, close.values, color="#f7931a", linewidth=0.8)
axes[0].set_ylabel("Close Price (USDT)")
axes[0].grid(True, alpha=0.3)

axes[1].bar(df.index, df["volume"], color="#777", width=0.03, alpha=0.7)
axes[1].set_ylabel("Volume")
axes[1].set_xlabel("Date (UTC+8)")
axes[1].grid(True, alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(OUT / "backtest_data.png", dpi=120, bbox_inches="tight")
plt.show()

## Strategy 1: SMA Crossover

In [ ]:
fast_period = 50   # 50h
slow_period = 200  # 200h

fast_ma = vbt.MA.run(close, fast_period, short_name="fast")
slow_ma = vbt.MA.run(close, slow_period, short_name="slow")

entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)

portfolio = vbt.Portfolio.from_signals(close, entries, exits, init_cash=10000)

print(f"=== SMA {fast_period}/{slow_period} Crossover ===")
print(f"Total Return: {portfolio.total_return():.2%}")
print(f"Sharpe Ratio: {portfolio.sharpe_ratio():.2f}")
print(f"Max Drawdown: {portfolio.max_drawdown():.2%}")
print(f"Win Rate: {portfolio.trades.win_rate():.2%}")
print(f"Total Trades: {portfolio.trades.count()}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(close.index, close.values, color="#f7931a", linewidth=0.8, label="Close")
ax.plot(fast_ma.ma.index, fast_ma.ma.values, color="blue", linewidth=0.6, label=f"SMA {fast_period}")
ax.plot(slow_ma.ma.index, slow_ma.ma.values, color="red", linewidth=0.6, label=f"SMA {slow_period}")

# Plot trade markers using raw entry/exit index arrays
trades = portfolio.trades
entry_idxs = trades.entry_idx.values
exit_idxs = trades.exit_idx.values
for i in range(len(entry_idxs)):
    ax.plot(close.index[entry_idxs[i]], close.iloc[entry_idxs[i]], "g^", markersize=6, alpha=0.7)
    ax.plot(close.index[exit_idxs[i]], close.iloc[exit_idxs[i]], "rv", markersize=6, alpha=0.7)

ax.set_title(f"SMA {fast_period}/{slow_period} Crossover — UTC+8")
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(OUT / "sma_crossover.png", dpi=120, bbox_inches="tight")
plt.show()

## Strategy 2: RSI Mean Reversion

In [ ]:
rsi_period = 14
oversold = 30
overbought = 70

rsi = vbt.RSI.run(close, rsi_period)
rsi_values = rsi.rsi

entries = rsi_values < oversold
exits = rsi_values > overbought

portfolio_rsi = vbt.Portfolio.from_signals(close, entries, exits, init_cash=10000)

print(f"=== RSI Mean Reversion ({rsi_period}) ===")
print(f"Total Return: {portfolio_rsi.total_return():.2%}")
print(f"Sharpe Ratio: {portfolio_rsi.sharpe_ratio():.2f}")
print(f"Max Drawdown: {portfolio_rsi.max_drawdown():.2%}")
print(f"Win Rate: {portfolio_rsi.trades.win_rate():.2%}")
print(f"Total Trades: {portfolio_rsi.trades.count()}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

ax1.plot(close.index, close.values, color="#f7931a", linewidth=0.8)
ax1.set_title(f"RSI Mean Reversion — UTC+8")
ax1.grid(True, alpha=0.3)

# Plot trade markers
trades = portfolio_rsi.trades
entry_idxs = trades.entry_idx.values
exit_idxs = trades.exit_idx.values
for i in range(len(entry_idxs)):
    ax1.plot(close.index[entry_idxs[i]], close.iloc[entry_idxs[i]], "g^", markersize=5, alpha=0.7)
    ax1.plot(close.index[exit_idxs[i]], close.iloc[exit_idxs[i]], "rv", markersize=5, alpha=0.7)

ax2.plot(rsi_values.index, rsi_values.values, color="purple", linewidth=0.6)
ax2.axhline(oversold, color="green", linestyle="--", alpha=0.5)
ax2.axhline(overbought, color="red", linestyle="--", alpha=0.5)
ax2.set_ylabel("RSI")
ax2.set_ylim(0, 100)
ax2.grid(True, alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(OUT / "rsi_reversion.png", dpi=120, bbox_inches="tight")
plt.show()

## Strategy 3: Bollinger Band Breakout

In [ ]:
bb_window = 20
bb_std = 2.0

bb = vbt.BBANDS.run(close, window=bb_window, alpha=bb_std)
upper = bb.upper
lower = bb.lower

entries = close > upper
exits = close < lower

portfolio_bb = vbt.Portfolio.from_signals(close, entries, exits, init_cash=10000)

print(f"=== Bollinger Band Breakout ({bb_window}, {bb_std} std) ===")
print(f"Total Return: {portfolio_bb.total_return():.2%}")
print(f"Sharpe Ratio: {portfolio_bb.sharpe_ratio():.2f}")
print(f"Max Drawdown: {portfolio_bb.max_drawdown():.2%}")
print(f"Win Rate: {portfolio_bb.trades.win_rate():.2%}")
print(f"Total Trades: {portfolio_bb.trades.count()}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(close.index, close.values, color="#f7931a", linewidth=0.8, label="Close")
ax.plot(upper.index, upper.values, color="gray", linewidth=0.5, alpha=0.6, label="Upper BB")
ax.plot(lower.index, lower.values, color="gray", linewidth=0.5, alpha=0.6, label="Lower BB")

# Plot trade markers
trades = portfolio_bb.trades
entry_idxs = trades.entry_idx.values
exit_idxs = trades.exit_idx.values
for i in range(len(entry_idxs)):
    ax.plot(close.index[entry_idxs[i]], close.iloc[entry_idxs[i]], "g^", markersize=5, alpha=0.7)
    ax.plot(close.index[exit_idxs[i]], close.iloc[exit_idxs[i]], "rv", markersize=5, alpha=0.7)

ax.set_title(f"Bollinger Band Breakout ({bb_window}, {bb_std} std) — UTC+8")
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(OUT / "bb_breakout.png", dpi=120, bbox_inches="tight")
plt.show()

## Strategy Comparison

In [ ]:
bh_return = (close.iloc[-1] / close.iloc[0]) - 1

# Build comparison from strategy stats
sma = [portfolio.total_return(), portfolio.sharpe_ratio(), portfolio.max_drawdown(), portfolio.trades.win_rate(), portfolio.trades.count()]
rsi = [portfolio_rsi.total_return(), portfolio_rsi.sharpe_ratio(), portfolio_rsi.max_drawdown(), portfolio_rsi.trades.win_rate(), portfolio_rsi.trades.count()]
bb = [portfolio_bb.total_return(), portfolio_bb.sharpe_ratio(), portfolio_bb.max_drawdown(), portfolio_bb.trades.win_rate(), portfolio_bb.trades.count()]

print("\n=== Strategy Comparison ===")
print(f"{'Metric':<15} │ {'SMA Crossover':>15} │ {'RSI Reversion':>15} │ {'BB Breakout':>15}")
print(f"{'─'*15}┼{'─'*22}┼{'─'*22}┼{'─'*22}")
print(f"{'Total Return':>15} │ {sma[0]:>15.2%} │ {rsi[0]:>15.2%} │ {bb[0]:>15.2%}")
print(f"{'Sharpe Ratio':>15} │ {sma[1]:>15.2f} │ {rsi[1]:>15.2f} │ {bb[1]:>15.2f}")
print(f"{'Max Drawdown':>15} │ {sma[2]:>15.2%} │ {rsi[2]:>15.2%} │ {bb[2]:>15.2%}")
print(f"{'Win Rate':>15} │ {sma[3]:>15.2%} │ {rsi[3]:>15.2%} │ {bb[3]:>15.2%}")
print(f"{'Total Trades':>15} │ {sma[4]:>15.0f} │ {rsi[4]:>15.0f} │ {bb[4]:>15.0f}")
print(f"{'─'*15}┼{'─'*22}┼{'─'*22}┼{'─'*22}")
print(f"{'Buy & Hold':>15} │ {bh_return:>15.2%} │ {'':>15} │ {'':>15}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Equity curves
ax.plot(portfolio.value().index, portfolio.value().values, label="SMA Crossover", alpha=0.8, linewidth=1)
ax.plot(portfolio_rsi.value().index, portfolio_rsi.value().values, label="RSI Reversion", alpha=0.8, linewidth=1)
ax.plot(portfolio_bb.value().index, portfolio_bb.value().values, label="BB Breakout", alpha=0.8, linewidth=1)
ax.axhline(10000, color="gray", linestyle="--", alpha=0.5, label="Initial Capital")
ax.set_title("Equity Curves — $10,000 Starting Capital | UTC+8")
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()

# Also save the comparison metrics plot
fig2, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ["Total Return", "Sharpe", "Max DD"]
bar_width = 0.25
x = np.arange(len(metrics))

strategy_data = {
    "SMA Crossover": [portfolio.total_return(), portfolio.sharpe_ratio(), portfolio.max_drawdown()],
    "RSI Reversion": [portfolio_rsi.total_return(), portfolio_rsi.sharpe_ratio(), portfolio_rsi.max_drawdown()],
    "BB Breakout": [portfolio_bb.total_return(), portfolio_bb.sharpe_ratio(), portfolio_bb.max_drawdown()],
}

colors = ["#f7931a", "#4ecdc4", "#a855f7"]
for i, (strategy, values) in enumerate(strategy_data.items()):
    axes[0].bar(x + i * bar_width, values, bar_width, label=strategy, color=colors[i])

axes[0].set_xticks(x + bar_width)
axes[0].set_xticklabels(metrics)
axes[0].set_title("Key Metrics Comparison")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Best strategy summary
best = max(strategy_data.items(), key=lambda x: x[1][0])
axes[1].text(0.5, 0.7, f"Best Return: {best[0]}", fontsize=16, ha='center')
axes[1].text(0.5, 0.5, f"{best[1][0]:.2%}", fontsize=14, ha='center')
axes[1].text(0.5, 0.3, f"vs Buy & Hold: {bh_return:.2%}", fontsize=12, ha='center', color='gray')
axes[1].set_title("Summary")
axes[1].axis('off')

plt.tight_layout()
plt.savefig(OUT / "strategy_comparison.png", dpi=120, bbox_inches="tight")
plt.show()